# Exercise — Run Governance as an Operation

Governance is ongoing, not a one-time project. **Trailhead Provisions** gives you a single
audit that rolls up every governance signal — missing catalog metadata, quality failures,
lineage gaps, consent conflict, and **access-control gaps**. Your job: run it, **remediate what
is fixable in AWS (the catalog in Glue and access in Lake Formation), and re-run until the
platform is compliant** — then write up what remains as ongoing watch items. See `INSTRUCTIONS.md`.

> Runs against live AWS Glue + Lake Formation when provisioned; local fallback offline.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog("trailhead.db")
print("Backend:", lf_backend(), "->", type(gc).__name__)

## 1. Run the platform governance audit (provided)

In [ ]:
findings = gc.governance_audit(contracts_valid=True)
print("Total findings:", len(findings))
findings.groupby(["signal", "severity"]).size().reset_index(name="count")

## 2. Remediate the catalog gaps in Glue
Close every `missing_metadata` finding by writing owner / classification / retention and tagging untagged columns (PII where the column holds personal data).

In [ ]:
# TODO(you): close every missing_metadata finding by writing it back to the catalog.
#   gc.set_table_metadata(table, owner=..., classification=..., retention=...)
#   gc.tag_column(table, column, classification)   # PII where the column holds personal data
print("Catalog gaps remaining:", len(gc.catalog_audit()))

## 3. Remediate the access-control gaps in Lake Formation
Every **PII table** must be governed by Lake Formation. Bring each PII-bearing table under an LF-Tag and grant the steward by tag.

In [ ]:
# TODO(you): for each PII-bearing table, bring it under Lake Formation
# (assign an LF-Tag and grant the steward by tag) so it is no longer an access_control_gap.
print("TODO: govern the PII tables above")

## 4. Re-audit and confirm compliance (provided)
The AWS-fixable findings — `missing_metadata` and `access_control_gap` — should both be zero. What remains are the recurring SLOs (quality, consent, lineage).

In [ ]:
re_findings = gc.governance_audit(contracts_valid=True)
def n(df, sig): return int((df["signal"] == sig).sum())
print("missing_metadata: ", n(findings, "missing_metadata"),  "->", n(re_findings, "missing_metadata"))
print("access_control_gap:", n(findings, "access_control_gap"), "->", n(re_findings, "access_control_gap"))
compliant = n(re_findings, "missing_metadata") == 0 and n(re_findings, "access_control_gap") == 0
print("AWS-fixable findings cleared?", compliant)   # True once you have remediated above
re_findings.groupby(["signal", "severity"]).size().reset_index(name="count")

## 5. Operations write-up
Replace the cell below with your write-up. Address every requirement in `INSTRUCTIONS.md`.

> **Your operations write-up here.** Address:
>
> - What you remediated in Glue (catalog) and in Lake Formation (access), and the before/after counts.
> - Why classifying PII columns surfaced new access-control obligations.
> - Which findings remain and why they are ongoing SLOs (quality, consent, lineage) rather than one-time fixes.
> - How you'd operate this audit continuously (schedule, ownership, trending).